In [1]:
import pandas as pd
import xarray as xr
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import glob
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [2]:
# 1. Pad naar jouw Parquet-bestand
# BASE_DIR = "/Users/gjwdijk/kedro/wf-kedro/data/01_raw/parquet/heart_rate_sample/"
# BASE_DIR = f'/Volumes/Extreme SSD/pq2/activity_file/'
BASE_DIR = f'/Volumes/SanDisk_ExtremePro_55AF/wf-kedro-dataset/'
DATA_TYPE_DIR = f'heart_rate_sample/'
#profile_id=9
#output_path = f"{BASE_DIR}profile_id={profile_id}/"
output_path = f"{BASE_DIR}{DATA_TYPE_DIR}"
output_path

'/Volumes/SanDisk_ExtremePro_55AF/wf-kedro-dataset/heart_rate_sample/'

In [3]:
# 2. Lees het Parquet-bestand in met Pandas
# We gaan ervan uit dat er kolommen zijn voor de tijd en eventuele andere dimensies/variabelen
files = glob.glob(f"{output_path}**/*.parquet", recursive=True)
files


[
    '/Volumes/SanDisk_ExtremePro_55AF/wf-kedro-dataset/heart_rate_sample/profile_id=15198138/part-0.parquet',
    '/Volumes/SanDisk_ExtremePro_55AF/wf-kedro-dataset/heart_rate_sample/profile_id=14768572/part-0.parquet',
    '/Volumes/SanDisk_ExtremePro_55AF/wf-kedro-dataset/heart_rate_sample/profile_id=7131855/part-0.parquet',
    '/Volumes/SanDisk_ExtremePro_55AF/wf-kedro-dataset/heart_rate_sample/profile_id=9796464/part-0.parquet',
    '/Volumes/SanDisk_ExtremePro_55AF/wf-kedro-dataset/heart_rate_sample/profile_id=18397769/part-0.parquet',
    '/Volumes/SanDisk_ExtremePro_55AF/wf-kedro-dataset/heart_rate_sample/profile_id=17884443/part-0.parquet',
    '/Volumes/SanDisk_ExtremePro_55AF/wf-kedro-dataset/heart_rate_sample/profile_id=13073159/part-0.parquet',
    '/Volumes/SanDisk_ExtremePro_55AF/wf-kedro-dataset/heart_rate_sample/profile_id=6568961/part-0.parquet',
    '/Volumes/SanDisk_ExtremePro_55AF/wf-kedro-dataset/heart_rate_sample/profile_id=15331446/part-0.parquet',
    '/Volu

In [4]:
len(files)

29684

In [5]:
dfs = []
n = 0
nt = len(files)
for f in files:
    print(f'{n}/{nt}')
    n += 1
    try:
        df_part = pq.read_table(f).to_pandas().drop(columns=['id', 'is_manual', 'min', 'max', 'value', 'duration'])
        if 'heart_rate_samples' in df_part.columns:
            # 1. Explodeer de kolom: elke sub-array (koppel) krijgt direct zijn eigen rij
            # Dit breekt de ongelijke rijen (van 15 en 60) direct supersnel af
            df_plat = df_part.explode('heart_rate_samples')
            
            # 2. Verwijder eventuele lege rijen (NaNs)
            df_plat = df_plat.dropna(subset=['heart_rate_samples'])
            
            if len(df_plat) > 0:
                # 3. Gebruik np.vstack om in 1x een snelle 2D NumPy-matrix te maken (C-snelheid)
                matrix = np.vstack(df_plat['heart_rate_samples'].values)
                
                # 4. Maak de kolommen direct aan door de matrix te snijden (slicing)
                # Dit kost nauwelijks tijd omdat er geen nieuwe objecten worden gebouwd
                df_plat['time_offset_in_seconds'] = matrix[:, 0]
                df_plat['bpm'] = matrix[:, 1]
                
                # 5. Gooi de oude object-kolom weg
                df_part = df_plat.drop(columns=['heart_rate_samples'])
                #print(df_part)
                dfs.append(df_part)
            else:
                print(f"XX - {f}")     
        else:
            print("YY")

    except Exception as e:
        print(f"Fout bij bestand {f}: {e}")
df = pd.concat(dfs, ignore_index=True)
df

0/29684
1/29684
2/29684
3/29684
XX - /Volumes/SanDisk_ExtremePro_55AF/wf-kedro-dataset/heart_rate_sample/profile_id=9796464/part-0.parquet
4/29684
5/29684
6/29684
7/29684
8/29684
XX - /Volumes/SanDisk_ExtremePro_55AF/wf-kedro-dataset/heart_rate_sample/profile_id=15331446/part-0.parquet
9/29684
10/29684
XX - /Volumes/SanDisk_ExtremePro_55AF/wf-kedro-dataset/heart_rate_sample/profile_id=3359905/part-0.parquet
11/29684
12/29684
XX - /Volumes/SanDisk_ExtremePro_55AF/wf-kedro-dataset/heart_rate_sample/profile_id=9881064/part-0.parquet
13/29684
XX - /Volumes/SanDisk_ExtremePro_55AF/wf-kedro-dataset/heart_rate_sample/profile_id=12714927/part-0.parquet
14/29684
XX - /Volumes/SanDisk_ExtremePro_55AF/wf-kedro-dataset/heart_rate_sample/profile_id=9317617/part-0.parquet
15/29684
16/29684
17/29684
18/29684
19/29684
20/29684
XX - /Volumes/SanDisk_ExtremePro_55AF/wf-kedro-dataset/heart_rate_sample/profile_id=17669083/part-0.parquet
21/29684
XX - /Volumes/SanDisk_ExtremePro_55AF/wf-kedro-dataset/heart

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:8                                                                                    │
│                                                                                                  │
│    5 │   print(f'{n}/{nt}')                                                                      │
│    6 │   n += 1                                                                                  │
│    7 │   try:                                                                                    │
│ ❱  8 │   │   df_part = pq.read_table(f).to_pandas().drop(columns=['id', 'is_manual', 'min', '    │
│    9 │   │   if 'heart_rate_samples' in df_part.columns:                                         │
│   10 │   │   │   # 1. Explodeer de kolom: elke sub-array (koppel) krijgt direct zijn eigen ri    │
│   11 │   │   │   # Dit breekt de ongelijke rijen (van 15 en 60) direct supersnel af              │
│                                                                                                  │
│ in pyarrow.lib._PandasConvertible.to_pandas:1071                                                 │
│                                                                                                  │
│ in pyarrow.lib.Table._to_pandas:5146                                                             │
│                                                                                                  │
│ /Users/gjwdijk/miniforge3/envs/wf-kedro/lib/python3.13/site-packages/pyarrow/pandas_compat.py:82 │
│ 2 in table_to_dataframe                                                                          │
│                                                                                                  │
│    819 │   columns = _deserialize_column_index(table, all_columns, column_indexes)               │
│    820 │                                                                                         │
│    821 │   column_names = table.column_names                                                     │
│ ❱  822 │   result = pa.lib.table_to_blocks(options, table, categories,                           │
│    823 │   │   │   │   │   │   │   │   │   list(ext_columns_dtypes.keys()))                      │
│    824 │   if _pandas_api.is_ge_v3():                                                            │
│    825 │   │   from pandas.api.internals import create_dataframe_from_blocks                     │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
KeyboardInterrupt

In [39]:
ts = df.iloc[0]
type(ts)
ts.time, ts.time_offset_in_seconds, ts.bpm

(Timestamp('2023-05-22 22:00:00'), np.int64(15), np.int64(71))

In [ ]:
# 1. Voorbeeld-dataopbouw (simulatie van jouw lijst met tuples)
# Stel dat 'raw_data' de lijst is die vol staat met de tuples zoals in jouw voorbeeld
raw_data = [
    (pd.Timestamp('2024-11-20 12:18:57'), np.array([[15, 60], [30, 60], [45, 60]], dtype=object)),
    # ... meer opeenvolgende timestamps ...
]

# 2. Extraheer de features en zorg voor een vaste vorm (shape)
# We halen de 2D-arrays los en stacken ze in een 3D-numpy matrix
X_list = []
timestamps = []

for timestamp, array_2d in raw_data:
    # Converteer de 'object' array naar zuivere floats/integers
    matrix_clean = np.array(array_2d.tolist(), dtype=np.float32)
    X_list.append(matrix_clean)
    timestamps.append(timestamp)

# Maak er één grote 3D matrix van
# Vorm wordt: (aantal_timestamps, aantal_metingen_per_timestamp, aantal_kolommen)
# In jouw voorbeeld: (samples, 59, 2)
X_3d = np.array(X_list)

# 3. Normalisatie (Schaal de data tussen 0 en 1 voor de LSTM)
# Omdat MinMaxScaler alleen 2D data accepteert, flatten we het tijdelijk
samples, time_steps, features = X_3d.shape
X_2d_flat = X_3d.reshape(-1, features)

scaler = MinMaxScaler()
X_2d_scaled = scaler.fit_transform(X_2d_flat)

# Schaal het direct weer terug naar de originele 3D LSTM-vorm
X_lstm = X_2d_scaled.reshape(samples, time_steps, features)

print("Definitieve vorm voor Keras LSTM (X):", X_lstm.shape)
# Output: (samples, 59, 2)
